# Data Fetch

This notebook documents how raw stock data was fetched from **Yahoo Finance** using the `yfinance` library.
The fetch logic lives in `deprecated/data_fetch.py` and is reproduced here for reference.

**Output files** (already present in `data/`):
- `data/SPY_raw.csv` — SPY daily OHLCV + adj_close
- `data/TSLA_raw.csv` — TSLA daily OHLCV + adj_close
- `data/VIX_raw.csv` — VIX daily OHLCV (volume = 0, adj_close = close)

**No API key required.** Yahoo Finance is free and supports up to 10 years of daily history.

---

## Configuration

| Parameter | Value |
|---|---|
| Tickers | `SPY`, `TSLA`, `^VIX` |
| Start date | 2015-01-01 |
| End date | 2024-12-31 |
| Frequency | Daily (1 trading day per row) |
| Source | Yahoo Finance via `yfinance` |

In [1]:
import yfinance as yf
import pandas as pd
import os

In [2]:
# ── Fetch configuration ───────────────────────────────────────────────────
TICKERS    = ["SPY", "TSLA", "^VIX"]
START_DATE = "2015-01-01"
END_DATE   = "2024-12-31"
DATA_DIR   = "data"


def download_data(ticker, start=START_DATE, end=END_DATE):
    """
    Download daily OHLCV data from Yahoo Finance.
    Free, no API key, full 10 years available.
    Uses Adj Close which handles splits and dividends.
    Note: VIX has no volume — filled with 0 for consistent schema.
    """
    print(f"  Downloading {ticker} ({start} to {end})...")

    df = yf.download(ticker, start=start, end=end, auto_adjust=False, progress=False)

    # Flatten MultiIndex columns if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # Rename columns to lowercase
    df.columns = [c.lower().replace(" ", "_") for c in df.columns]

    # Handle adj close naming
    if "adj close" in df.columns:
        df.rename(columns={"adj close": "adj_close"}, inplace=True)

    # VIX has no volume or adj_close — fill with 0 for consistent schema
    if "volume" not in df.columns:
        df["volume"] = 0
    if "adj_close" not in df.columns:
        df["adj_close"] = df["close"]

    # Keep only OHLCV + Adj Close
    cols = ["open", "high", "low", "close", "adj_close", "volume"]
    df   = df[[c for c in cols if c in df.columns]].copy()

    df.index      = pd.to_datetime(df.index)
    df.index.name = "date"
    df.sort_index(inplace=True)

    return df


def save_data(df, ticker):
    os.makedirs(DATA_DIR, exist_ok=True)
    clean_name = ticker.replace("^", "")   # ^VIX → VIX
    path = os.path.join(DATA_DIR, f"{clean_name}_raw.csv")
    df.to_csv(path)
    print(f"  Saved to {path}")

## Re-fetch Data (run only if refreshing)

The cell below re-downloads and overwrites the CSVs in `data/`. **Skip this cell** if the existing files are current — the data is already cached and there is no need to re-fetch unless the date range changes.

In [3]:
# ── Uncomment to re-fetch all tickers ────────────────────────────────────
# for ticker in TICKERS:
#     print(f"\n--- {ticker} ---")
#     df = download_data(ticker)
#     save_data(df, ticker)
# print("\nAll done.")

print("Fetch skipped — using existing cached files in data/")

Fetch skipped — using existing cached files in data/


## Verify Cached Data

In [4]:
# ── Load and preview all three raw CSVs ──────────────────────────────────
files = {
    "SPY":  "data/SPY_raw.csv",
    "TSLA": "data/TSLA_raw.csv",
    "VIX":  "data/VIX_raw.csv",
}

for ticker, path in files.items():
    df = pd.read_csv(path, parse_dates=["date"], index_col="date")
    print(f"{'='*50}")
    print(f"  {ticker}")
    print(f"  Rows:    {len(df):,}")
    print(f"  Range:   {df.index[0].date()} \u2192 {df.index[-1].date()}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Missing: {df.isnull().sum().sum()} total NaN values")
    print()
    print(df.head(3).to_string())
    print()

  SPY
  Rows:    2,515
  Range:   2015-01-02 → 2024-12-30
  Columns: ['open', 'high', 'low', 'close', 'adj_close', 'volume']
  Missing: 0 total NaN values

                  open        high         low       close   adj_close     volume
date                                                                             
2015-01-02  206.380005  206.880005  204.179993  205.429993  170.125015  121465900
2015-01-05  204.169998  204.369995  201.350006  201.720001  167.052597  169632600
2015-01-06  202.089996  202.720001  198.860001  199.820007  165.479126  209151400

  TSLA
  Rows:    2,515
  Range:   2015-01-02 → 2024-12-30
  Columns: ['open', 'high', 'low', 'close', 'adj_close', 'volume']
  Missing: 0 total NaN values

                 open       high        low      close  adj_close    volume
date                                                                       
2015-01-02  14.858000  14.883333  14.217333  14.620667  14.620667  71466000
2015-01-05  14.303333  14.433333  13.810667  14.